In [1]:
import pyclifford as pc
import mcp_server as mcp
import numpy
from math import ceil

def sample_random_pauli_string(N):
    """Sample a random Pauli operator string with random phase for N qubits.
    Returns a string like '-iIXIIYZXIZ'.
    """
    paulis = ['I', 'X', 'Y', 'Z']
    phases = ['', 'i', '-', '-i']
    pauli_str = ''.join(numpy.random.choice(paulis) for _ in range(N))
    phase = numpy.random.choice(phases)
    return f'{phase}{pauli_str}'

def sample_question_and_answer(N):
    pauli_str1 = sample_random_pauli_string(N)
    pauli_str2 = sample_random_pauli_string(N)
    op1 = pc.pauli(pauli_str1)
    op2 = pc.pauli(pauli_str2)
    str1 = mcp.PauliTerm.from_obj(op1).text
    str2 = mcp.PauliTerm.from_obj(op2).text
    sol = mcp.PauliTerm.from_obj(op1 @ op2)
    return f'({str1}) ({str2})', sol

N_max = 10
iterations = 10


def generate_questions_and_answers(N_max, iterations):
    questions = []
    answers = []
    for i in range(iterations):
        q, a = sample_question_and_answer(N_max)
        questions.append(q)
        answers.append(a)

    questions_block = "\n".join(
        [f"({i+1}) Input: $ {q} $\n    Your Answer: " for i, q in enumerate(questions)]
    )

    pauli_instruction_template = f"""You are given two strings of Pauli operators acting on qubits indexed by subscripts. Each operator string may also include a global phase factor from the set {{±1,±i}}. The Pauli operators obey the following multiplication rules:

    X_{{j}}^2 = Y_{{j}}^2 = Z_{{j}}^2 = I  
    X_{{j}}Y_{{j}} = iZ_{{j}}, Y_{{j}}Z_{{j}} = iX_{{j}}, Z_{{j}}X_{{j}} = iY_{{j}} (and cyclic permutations)  
    Operators on the same qubit anticommute.  
    Operators on different qubits commute.  

    Your task is to compute the product of the two operator strings by applying the Pauli algebra. The result should be a single simplified operator string, combining operators acting on the same qubit where applicable, and simplifying all coefficients to a single phase factor (±1, ±i). You must preserve the order of qubit indices in ascending order in your final expression.

    Format your answer as a single operator string, with phase factor in front (e.g., -i X_{{1}}Y_{{2}}Z_{{3}}), omitting identity operators. Do not include explanatory text or steps. Use LaTeX-style syntax, e.g., Z_{{1}}, X_{{2}} etc., and keep one space between operators.

    Examples:

    (1) Input: $(+iZ_{{1}})(−iX_{{1}})$  
        Answer: +iY_{{1}}

    (2) Input: $(+iZ_{{1}}Y_{{2}})(Z_{{1}}X_{{2}})$  
        Answer: Z_{{2}}

    Questions (Compute and give your answer in the same format as above):

    {questions_block}

    ---

    After answering all questions, output your answers as a Python list of strings for easy copying and checking.

    Please follow this exact format (note: this is an example, not related to the above questions):

    ```python
    LLM_answers = [
        '-i Y_{0}',
        '- X_{0} Z_{1}',
        'X_{0} Y_{1} Y_{2}',
        '- Y_{1} Z_{2} Y_{3}',
        '- Y_{0} X_{1} Y_{4}',
        '-i Z_{2} Z_{3} Z_{4} Z_{5} X_{6}',
        '+i Y_{0} Z_{1} Y_{2} X_{3} X_{6}',
        '+i Z_{0} Z_{1} Y_{2} Y_{3} Y_{4} Z_{6} Y_{8}',
        'Z_{1} X_{5} Z_{6} X_{7} Y_{8}',
        ...
    ]"""

    return pauli_instruction_template, answers

## Without MCP tools:

### ChatGPT

#### 4o model

In [2]:
N_max = 1
iterations = 10

questions, answers = generate_questions_and_answers(N_max, iterations)
print(questions)

You are given two strings of Pauli operators acting on qubits indexed by subscripts. Each operator string may also include a global phase factor from the set {±1,±i}. The Pauli operators obey the following multiplication rules:

    X_{j}^2 = Y_{j}^2 = Z_{j}^2 = I  
    X_{j}Y_{j} = iZ_{j}, Y_{j}Z_{j} = iX_{j}, Z_{j}X_{j} = iY_{j} (and cyclic permutations)  
    Operators on the same qubit anticommute.  
    Operators on different qubits commute.  

    Your task is to compute the product of the two operator strings by applying the Pauli algebra. The result should be a single simplified operator string, combining operators acting on the same qubit where applicable, and simplifying all coefficients to a single phase factor (±1, ±i). You must preserve the order of qubit indices in ascending order in your final expression.

    Format your answer as a single operator string, with phase factor in front (e.g., -i X_{1}Y_{2}Z_{3}), omitting identity operators. Do not include explanatory text

In [3]:
answers

[PauliTerm(coefficient=1j, pauli_string={0: 'Z'}, text='+i Z_{0}'),
 PauliTerm(coefficient=(1+0j), pauli_string={}, text='I'),
 PauliTerm(coefficient=(1+0j), pauli_string={0: 'Y'}, text='Y_{0}'),
 PauliTerm(coefficient=(-1+0j), pauli_string={0: 'Z'}, text='- Z_{0}'),
 PauliTerm(coefficient=(-1+0j), pauli_string={0: 'Y'}, text='- Y_{0}'),
 PauliTerm(coefficient=(-1+0j), pauli_string={0: 'Y'}, text='- Y_{0}'),
 PauliTerm(coefficient=(-1+0j), pauli_string={}, text='- I'),
 PauliTerm(coefficient=(1+0j), pauli_string={0: 'X'}, text='X_{0}'),
 PauliTerm(coefficient=(-0-1j), pauli_string={0: 'Y'}, text='-i Y_{0}'),
 PauliTerm(coefficient=(1+0j), pauli_string={0: 'X'}, text='X_{0}')]

In [4]:
LLM_answers = [
    '+i Z_{0}',
    '-1',
    '+i Y_{0}',
    '+i Z_{0}',
    '-1 Y_{0}',
    '-i Y_{0}',
    '-1',
    '+ X_{0}',
    '- Y_{0}',
    '+ X_{0}'
]

In [5]:
LLM_answers_pc = [mcp.PauliTerm(text = LLM_answers[i]) for i in range(len(LLM_answers))]
accuracy = [LLM_answers_pc[i] == answers[i] for i in range(min(len(LLM_answers), len(answers)))]
print(f'N_max = {N_max}, question_volume  = {iterations}, accuracy = {sum(accuracy)/len(accuracy)}')

N_max = 1, question_volume  = 10, accuracy = 0.5


In [6]:
accuracy

[True, False, False, False, True, False, True, True, False, True]

________________________________________

In [7]:
N_max = 2
iterations = 10

questions, answers = generate_questions_and_answers(N_max, iterations)
print(questions)

You are given two strings of Pauli operators acting on qubits indexed by subscripts. Each operator string may also include a global phase factor from the set {±1,±i}. The Pauli operators obey the following multiplication rules:

    X_{j}^2 = Y_{j}^2 = Z_{j}^2 = I  
    X_{j}Y_{j} = iZ_{j}, Y_{j}Z_{j} = iX_{j}, Z_{j}X_{j} = iY_{j} (and cyclic permutations)  
    Operators on the same qubit anticommute.  
    Operators on different qubits commute.  

    Your task is to compute the product of the two operator strings by applying the Pauli algebra. The result should be a single simplified operator string, combining operators acting on the same qubit where applicable, and simplifying all coefficients to a single phase factor (±1, ±i). You must preserve the order of qubit indices in ascending order in your final expression.

    Format your answer as a single operator string, with phase factor in front (e.g., -i X_{1}Y_{2}Z_{3}), omitting identity operators. Do not include explanatory text

In [8]:
answers

[PauliTerm(coefficient=(-0-1j), pauli_string={1: 'X'}, text='-i X_{1}'),
 PauliTerm(coefficient=1j, pauli_string={0: 'Z', 1: 'X'}, text='+i Z_{0} X_{1}'),
 PauliTerm(coefficient=(-1+0j), pauli_string={0: 'X'}, text='- X_{0}'),
 PauliTerm(coefficient=(-0-1j), pauli_string={0: 'X', 1: 'Z'}, text='-i X_{0} Z_{1}'),
 PauliTerm(coefficient=(-1+0j), pauli_string={0: 'Y', 1: 'Y'}, text='- Y_{0} Y_{1}'),
 PauliTerm(coefficient=1j, pauli_string={}, text='+i I'),
 PauliTerm(coefficient=1j, pauli_string={1: 'Y'}, text='+i Y_{1}'),
 PauliTerm(coefficient=(1+0j), pauli_string={0: 'X', 1: 'Y'}, text='X_{0} Y_{1}'),
 PauliTerm(coefficient=(-0-1j), pauli_string={0: 'Y', 1: 'Y'}, text='-i Y_{0} Y_{1}'),
 PauliTerm(coefficient=(-1+0j), pauli_string={}, text='- I')]

In [9]:
LLM_answers = [
    '-i X_{1}',
    '+i Z_{0} X_{1}',
    '- X_{1}',
    '-i X_{0} Z_{1}',
    '+ X_{0} Y_{1}',
    '+1',
    '+i Y_{1}',
    '-i X_{0} X_{1} Z_{1}',
    '+ Z_{0} Y_{1}',
    '-1'
]

In [10]:
LLM_answers_pc = [mcp.PauliTerm(text = LLM_answers[i]) for i in range(len(LLM_answers))]
accuracy = [LLM_answers_pc[i] == answers[i] for i in range(min(len(LLM_answers), len(answers)))]
print(f'N_max = {N_max}, question_volume  = {iterations}, accuracy = {sum(accuracy)/len(accuracy)}')

N_max = 2, question_volume  = 10, accuracy = 0.5


In [11]:
accuracy

[True, True, False, True, False, False, True, False, False, True]

________________________________________

In [12]:
N_max = 3
iterations = 10

questions, answers = generate_questions_and_answers(N_max, iterations)
print(questions)

You are given two strings of Pauli operators acting on qubits indexed by subscripts. Each operator string may also include a global phase factor from the set {±1,±i}. The Pauli operators obey the following multiplication rules:

    X_{j}^2 = Y_{j}^2 = Z_{j}^2 = I  
    X_{j}Y_{j} = iZ_{j}, Y_{j}Z_{j} = iX_{j}, Z_{j}X_{j} = iY_{j} (and cyclic permutations)  
    Operators on the same qubit anticommute.  
    Operators on different qubits commute.  

    Your task is to compute the product of the two operator strings by applying the Pauli algebra. The result should be a single simplified operator string, combining operators acting on the same qubit where applicable, and simplifying all coefficients to a single phase factor (±1, ±i). You must preserve the order of qubit indices in ascending order in your final expression.

    Format your answer as a single operator string, with phase factor in front (e.g., -i X_{1}Y_{2}Z_{3}), omitting identity operators. Do not include explanatory text

In [13]:
answers

[PauliTerm(coefficient=(-0-1j), pauli_string={0: 'Z', 1: 'X'}, text='-i Z_{0} X_{1}'),
 PauliTerm(coefficient=1j, pauli_string={0: 'X', 1: 'Z', 2: 'Y'}, text='+i X_{0} Z_{1} Y_{2}'),
 PauliTerm(coefficient=(-0-1j), pauli_string={0: 'Z', 1: 'Z'}, text='-i Z_{0} Z_{1}'),
 PauliTerm(coefficient=(1+0j), pauli_string={0: 'Z', 1: 'Z', 2: 'Y'}, text='Z_{0} Z_{1} Y_{2}'),
 PauliTerm(coefficient=1j, pauli_string={2: 'Z'}, text='+i Z_{2}'),
 PauliTerm(coefficient=(1+0j), pauli_string={0: 'Y', 1: 'Y', 2: 'Y'}, text='Y_{0} Y_{1} Y_{2}'),
 PauliTerm(coefficient=1j, pauli_string={0: 'Y'}, text='+i Y_{0}'),
 PauliTerm(coefficient=(-1+0j), pauli_string={0: 'Y', 1: 'X'}, text='- Y_{0} X_{1}'),
 PauliTerm(coefficient=(-0-1j), pauli_string={0: 'X', 1: 'Z', 2: 'Y'}, text='-i X_{0} Z_{1} Y_{2}'),
 PauliTerm(coefficient=(1+0j), pauli_string={1: 'Z'}, text='Z_{1}')]

In [14]:
LLM_answers = [
    '- Y_{1} Z_{0}',
    '+i X_{0} Y_{1} X_{2} Y_{0}',
    '+i Z_{0} Z_{1}',
    '+ X_{2} Z_{0} Z_{1} Z_{2}',
    '+i Z_{2}',
    '+i Y_{0} X_{1} Y_{1} Y_{2} Z_{2}',
    '+ Y_{0}',
    '-i X_{1} Y_{0}',
    '- Z_{0} Y_{0} Z_{1} Y_{2}',
    '+i X_{1} Y_{1}'
]

In [15]:
LLM_answers_pc = [mcp.PauliTerm(text = LLM_answers[i]) for i in range(len(LLM_answers))]
accuracy = [LLM_answers_pc[i] == answers[i] for i in range(min(len(LLM_answers), len(answers)))]
print(f'N_max = {N_max}, question_volume  = {iterations}, accuracy = {sum(accuracy)/len(accuracy)}')

N_max = 3, question_volume  = 10, accuracy = 0.1


In [16]:
accuracy

[False, False, False, False, True, False, False, False, False, False]

________________________________________

In [17]:
N_max = 4
iterations = 10

questions, answers = generate_questions_and_answers(N_max, iterations)
print(questions)

You are given two strings of Pauli operators acting on qubits indexed by subscripts. Each operator string may also include a global phase factor from the set {±1,±i}. The Pauli operators obey the following multiplication rules:

    X_{j}^2 = Y_{j}^2 = Z_{j}^2 = I  
    X_{j}Y_{j} = iZ_{j}, Y_{j}Z_{j} = iX_{j}, Z_{j}X_{j} = iY_{j} (and cyclic permutations)  
    Operators on the same qubit anticommute.  
    Operators on different qubits commute.  

    Your task is to compute the product of the two operator strings by applying the Pauli algebra. The result should be a single simplified operator string, combining operators acting on the same qubit where applicable, and simplifying all coefficients to a single phase factor (±1, ±i). You must preserve the order of qubit indices in ascending order in your final expression.

    Format your answer as a single operator string, with phase factor in front (e.g., -i X_{1}Y_{2}Z_{3}), omitting identity operators. Do not include explanatory text

In [18]:
answers

[PauliTerm(coefficient=1j, pauli_string={1: 'X', 2: 'Y'}, text='+i X_{1} Y_{2}'),
 PauliTerm(coefficient=(-0-1j), pauli_string={0: 'X', 2: 'Z'}, text='-i X_{0} Z_{2}'),
 PauliTerm(coefficient=(-1+0j), pauli_string={0: 'X', 1: 'X', 2: 'Y', 3: 'Z'}, text='- X_{0} X_{1} Y_{2} Z_{3}'),
 PauliTerm(coefficient=(-0-1j), pauli_string={0: 'X', 1: 'Y', 2: 'X'}, text='-i X_{0} Y_{1} X_{2}'),
 PauliTerm(coefficient=(-1+0j), pauli_string={0: 'X', 1: 'Z'}, text='- X_{0} Z_{1}'),
 PauliTerm(coefficient=(-1+0j), pauli_string={0: 'Z', 2: 'Y', 3: 'Z'}, text='- Z_{0} Y_{2} Z_{3}'),
 PauliTerm(coefficient=(-0-1j), pauli_string={0: 'Z', 2: 'X', 3: 'Z'}, text='-i Z_{0} X_{2} Z_{3}'),
 PauliTerm(coefficient=(-0-1j), pauli_string={2: 'Z'}, text='-i Z_{2}'),
 PauliTerm(coefficient=(-0-1j), pauli_string={0: 'Y', 1: 'Z', 2: 'X', 3: 'Y'}, text='-i Y_{0} Z_{1} X_{2} Y_{3}'),
 PauliTerm(coefficient=1j, pauli_string={0: 'X', 1: 'X', 2: 'Z', 3: 'X'}, text='+i X_{0} X_{1} Z_{2} X_{3}')]

In [19]:
LLM_answers = [
    '+i Z_{1} Y_{2} X_{2}',
    '+ X_{0} Z_{1} Z_{2}',
    '- X_{0} X_{1} Y_{2} Z_{3}',
    '+ Z_{0} Y_{0} Y_{1} X_{2}',
    '-i X_{0} Z_{1} X_{1}',
    '- Y_{0} X_{1} Y_{2} Z_{2} Z_{3}',
    '-i X_{2} Z_{0} Z_{3}',
    '-i Z_{2}',
    '-i Y_{0} Z_{0} Z_{1} X_{2} Y_{3}',
    '- Y_{0} Z_{1} Z_{2} X_{3}'
]

In [20]:
LLM_answers_pc = [mcp.PauliTerm(text = LLM_answers[i]) for i in range(len(LLM_answers))]
accuracy = [LLM_answers_pc[i] == answers[i] for i in range(min(len(LLM_answers), len(answers)))]
print(f'N_max = {N_max}, question_volume  = {iterations}, accuracy = {sum(accuracy)/len(accuracy)}')

N_max = 4, question_volume  = 10, accuracy = 0.2


In [21]:
accuracy

[False, False, True, False, False, False, False, True, False, False]

________________________________________

In [22]:
N_max = 5
iterations = 10

questions, answers = generate_questions_and_answers(N_max, iterations)
print(questions)

You are given two strings of Pauli operators acting on qubits indexed by subscripts. Each operator string may also include a global phase factor from the set {±1,±i}. The Pauli operators obey the following multiplication rules:

    X_{j}^2 = Y_{j}^2 = Z_{j}^2 = I  
    X_{j}Y_{j} = iZ_{j}, Y_{j}Z_{j} = iX_{j}, Z_{j}X_{j} = iY_{j} (and cyclic permutations)  
    Operators on the same qubit anticommute.  
    Operators on different qubits commute.  

    Your task is to compute the product of the two operator strings by applying the Pauli algebra. The result should be a single simplified operator string, combining operators acting on the same qubit where applicable, and simplifying all coefficients to a single phase factor (±1, ±i). You must preserve the order of qubit indices in ascending order in your final expression.

    Format your answer as a single operator string, with phase factor in front (e.g., -i X_{1}Y_{2}Z_{3}), omitting identity operators. Do not include explanatory text

In [23]:
answers

[PauliTerm(coefficient=1j, pauli_string={0: 'X', 2: 'Y', 3: 'Y', 4: 'X'}, text='+i X_{0} Y_{2} Y_{3} X_{4}'),
 PauliTerm(coefficient=(-1+0j), pauli_string={0: 'Z', 1: 'Y', 2: 'Z', 3: 'Z', 4: 'Y'}, text='- Z_{0} Y_{1} Z_{2} Z_{3} Y_{4}'),
 PauliTerm(coefficient=1j, pauli_string={0: 'Y', 1: 'Z', 2: 'Z'}, text='+i Y_{0} Z_{1} Z_{2}'),
 PauliTerm(coefficient=(-0-1j), pauli_string={0: 'Z', 1: 'Y', 2: 'Z', 3: 'X', 4: 'Z'}, text='-i Z_{0} Y_{1} Z_{2} X_{3} Z_{4}'),
 PauliTerm(coefficient=(-1+0j), pauli_string={0: 'X', 1: 'X', 3: 'Z', 4: 'Z'}, text='- X_{0} X_{1} Z_{3} Z_{4}'),
 PauliTerm(coefficient=(-1+0j), pauli_string={1: 'Y', 2: 'Z', 3: 'X', 4: 'Y'}, text='- Y_{1} Z_{2} X_{3} Y_{4}'),
 PauliTerm(coefficient=(1+0j), pauli_string={0: 'Z', 1: 'Z', 2: 'X', 3: 'Y', 4: 'Z'}, text='Z_{0} Z_{1} X_{2} Y_{3} Z_{4}'),
 PauliTerm(coefficient=(1+0j), pauli_string={1: 'X', 2: 'Z', 3: 'Y', 4: 'X'}, text='X_{1} Z_{2} Y_{3} X_{4}'),
 PauliTerm(coefficient=1j, pauli_string={0: 'Y', 2: 'Y'}, text='+i Y_{0} 

In [24]:
LLM_answers = [
    '- X_{0} Y_{2} Y_{3} X_{4}',
    '+ Z_{0} Y_{1} Z_{2} Z_{3} Y_{4}',
    '+ Z_{0} X_{0} Z_{1} Z_{2} Y_{2}',
    '- Z_{0} Y_{1} Z_{2} Y_{3} Z_{4}',
    '-i X_{0} X_{1} Z_{1} Z_{3}',
    '+i Y_{1} Z_{2} X_{3} Y_{3} Y_{4}',
    '- Z_{1} X_{2} Y_{3} Z_{3} Z_{4}',
    '- Z_{1} Z_{2} Y_{3}',
    '+ Y_{0} Z_{2}',
    '+i Y_{0} X_{1} Y_{3}'
]

In [25]:
LLM_answers_pc = [mcp.PauliTerm(text = LLM_answers[i]) for i in range(len(LLM_answers))]
accuracy = [LLM_answers_pc[i] == answers[i] for i in range(min(len(LLM_answers), len(answers)))]
print(f'N_max = {N_max}, question_volume  = {iterations}, accuracy = {sum(accuracy)/len(accuracy)}')

N_max = 5, question_volume  = 10, accuracy = 0.0


In [26]:
accuracy

[False, False, False, False, False, False, False, False, False, False]

________________________________________

In [27]:
N_max = 6
iterations = 10

questions, answers = generate_questions_and_answers(N_max, iterations)
print(questions)

You are given two strings of Pauli operators acting on qubits indexed by subscripts. Each operator string may also include a global phase factor from the set {±1,±i}. The Pauli operators obey the following multiplication rules:

    X_{j}^2 = Y_{j}^2 = Z_{j}^2 = I  
    X_{j}Y_{j} = iZ_{j}, Y_{j}Z_{j} = iX_{j}, Z_{j}X_{j} = iY_{j} (and cyclic permutations)  
    Operators on the same qubit anticommute.  
    Operators on different qubits commute.  

    Your task is to compute the product of the two operator strings by applying the Pauli algebra. The result should be a single simplified operator string, combining operators acting on the same qubit where applicable, and simplifying all coefficients to a single phase factor (±1, ±i). You must preserve the order of qubit indices in ascending order in your final expression.

    Format your answer as a single operator string, with phase factor in front (e.g., -i X_{1}Y_{2}Z_{3}), omitting identity operators. Do not include explanatory text

In [28]:
answers

[PauliTerm(coefficient=1j, pauli_string={0: 'Y', 2: 'X', 3: 'Z', 4: 'Z'}, text='+i Y_{0} X_{2} Z_{3} Z_{4}'),
 PauliTerm(coefficient=(-1+0j), pauli_string={3: 'Z', 4: 'X', 5: 'Z'}, text='- Z_{3} X_{4} Z_{5}'),
 PauliTerm(coefficient=(-0-1j), pauli_string={0: 'Z', 1: 'Z', 2: 'Y', 4: 'Y', 5: 'Y'}, text='-i Z_{0} Z_{1} Y_{2} Y_{4} Y_{5}'),
 PauliTerm(coefficient=(-1+0j), pauli_string={0: 'Z', 2: 'Y', 3: 'Y', 4: 'Z'}, text='- Z_{0} Y_{2} Y_{3} Z_{4}'),
 PauliTerm(coefficient=(-0-1j), pauli_string={0: 'Y', 2: 'Z', 4: 'X', 5: 'X'}, text='-i Y_{0} Z_{2} X_{4} X_{5}'),
 PauliTerm(coefficient=(-1+0j), pauli_string={0: 'X', 1: 'Y', 2: 'X', 3: 'Y', 4: 'X'}, text='- X_{0} Y_{1} X_{2} Y_{3} X_{4}'),
 PauliTerm(coefficient=(-1+0j), pauli_string={0: 'Y', 1: 'X', 2: 'Y', 3: 'X', 4: 'Z'}, text='- Y_{0} X_{1} Y_{2} X_{3} Z_{4}'),
 PauliTerm(coefficient=(-1+0j), pauli_string={0: 'Y', 1: 'Z', 2: 'Z', 5: 'Y'}, text='- Y_{0} Z_{1} Z_{2} Y_{5}'),
 PauliTerm(coefficient=1j, pauli_string={0: 'X', 1: 'Y', 2: 'Z

In [29]:
LLM_answers = [
    '+i X_{0} Y_{2} Y_{5} Z_{0} Z_{2} Z_{3} Z_{4}',
    '+ Z_{3} X_{4} Z_{5}',
    '-i X_{0} Y_{0} Z_{1} Z_{2} X_{2} Y_{4}',
    '- X_{0} Y_{0} Y_{2} Y_{3} Z_{3} Z_{4}',
    '+ X_{4} X_{5} Z_{2}',
    '+ Y_{0} Z_{1} X_{1} X_{2} X_{3} X_{4}',
    '+i Y_{0} X_{1} Y_{2} X_{3} Z_{4}',
    '- Z_{2} Z_{5} Y_{0} Z_{1} Z_{3} Y_{4}',
    '+ Z_{0} Y_{0} Y_{1} X_{2} Y_{2} Y_{5}',
    '- X_{0} Z_{1} X_{3} Y_{1}'
]

In [30]:
LLM_answers_pc = [mcp.PauliTerm(text = LLM_answers[i]) for i in range(len(LLM_answers))]
accuracy = [LLM_answers_pc[i] == answers[i] for i in range(min(len(LLM_answers), len(answers)))]
print(f'N_max = {N_max}, question_volume  = {iterations}, accuracy = {sum(accuracy)/len(accuracy)}')

N_max = 6, question_volume  = 10, accuracy = 0.0


In [31]:
accuracy

[False, False, False, False, False, False, False, False, False, False]

________________________________________

In [33]:
N_max = 7
iterations = 10

questions, answers = generate_questions_and_answers(N_max, iterations)
print(questions)

You are given two strings of Pauli operators acting on qubits indexed by subscripts. Each operator string may also include a global phase factor from the set {±1,±i}. The Pauli operators obey the following multiplication rules:

    X_{j}^2 = Y_{j}^2 = Z_{j}^2 = I  
    X_{j}Y_{j} = iZ_{j}, Y_{j}Z_{j} = iX_{j}, Z_{j}X_{j} = iY_{j} (and cyclic permutations)  
    Operators on the same qubit anticommute.  
    Operators on different qubits commute.  

    Your task is to compute the product of the two operator strings by applying the Pauli algebra. The result should be a single simplified operator string, combining operators acting on the same qubit where applicable, and simplifying all coefficients to a single phase factor (±1, ±i). You must preserve the order of qubit indices in ascending order in your final expression.

    Format your answer as a single operator string, with phase factor in front (e.g., -i X_{1}Y_{2}Z_{3}), omitting identity operators. Do not include explanatory text

In [34]:
answers

[PauliTerm(coefficient=1j, pauli_string={0: 'Z', 1: 'X', 2: 'Y', 4: 'Y', 5: 'Y'}, text='+i Z_{0} X_{1} Y_{2} Y_{4} Y_{5}'),
 PauliTerm(coefficient=1j, pauli_string={0: 'Z', 1: 'Z', 2: 'Y', 4: 'Y', 6: 'Z'}, text='+i Z_{0} Z_{1} Y_{2} Y_{4} Z_{6}'),
 PauliTerm(coefficient=1j, pauli_string={0: 'X', 4: 'Z', 5: 'Y', 6: 'Y'}, text='+i X_{0} Z_{4} Y_{5} Y_{6}'),
 PauliTerm(coefficient=(-0-1j), pauli_string={0: 'X', 1: 'Y', 2: 'Z', 3: 'Z', 5: 'Z', 6: 'Y'}, text='-i X_{0} Y_{1} Z_{2} Z_{3} Z_{5} Y_{6}'),
 PauliTerm(coefficient=(1+0j), pauli_string={0: 'Z', 1: 'X', 2: 'Y', 3: 'Z', 4: 'Y', 5: 'Y'}, text='Z_{0} X_{1} Y_{2} Z_{3} Y_{4} Y_{5}'),
 PauliTerm(coefficient=(1+0j), pauli_string={0: 'X', 1: 'Z', 2: 'X', 4: 'Y', 5: 'X', 6: 'Z'}, text='X_{0} Z_{1} X_{2} Y_{4} X_{5} Z_{6}'),
 PauliTerm(coefficient=(-0-1j), pauli_string={0: 'Z', 1: 'Y', 2: 'Y', 5: 'X', 6: 'Z'}, text='-i Z_{0} Y_{1} Y_{2} X_{5} Z_{6}'),
 PauliTerm(coefficient=(1+0j), pauli_string={0: 'X', 1: 'X', 2: 'Z', 3: 'Z', 4: 'X', 5: 'Y',

In [35]:
LLM_answers = [
    '+ Z_{0} X_{1} Y_{2} Y_{4} Y_{5}',
    '+i Z_{0} Z_{1} Y_{2} Y_{4} X_{6}',
    '+ X_{0} X_{1} I X_{5} Y_{6}',
    '+i Z_{0} Y_{1} Z_{2} Y_{4} Z_{5} Y_{6}',
    '+ Z_{0} X_{1} X_{2} Z_{2} Y_{5} Z_{6}',
    '+i X_{0} Z_{2} Y_{4}',
    '- Y_{0} X_{0} Y_{1} X_{2} Y_{5}',
    '+i X_{1} Z_{2} Z_{3} X_{4} Y_{5} Y_{6}',
    '+ Y_{0} Z_{0} X_{1} X_{2} X_{3} Y_{3} Y_{4} Z_{6}',
    '+ Y_{0} Y_{1} Y_{3} Y_{4} Y_{5} X_{6}'
]

In [36]:
LLM_answers_pc = [mcp.PauliTerm(text = LLM_answers[i]) for i in range(len(LLM_answers))]
accuracy = [LLM_answers_pc[i] == answers[i] for i in range(min(len(LLM_answers), len(answers)))]
print(f'N_max = {N_max}, question_volume  = {iterations}, accuracy = {sum(accuracy)/len(accuracy)}')

N_max = 7, question_volume  = 10, accuracy = 0.0


________________________________________

In [37]:
N_max = 8
iterations = 10

questions, answers = generate_questions_and_answers(N_max, iterations)
print(questions)

You are given two strings of Pauli operators acting on qubits indexed by subscripts. Each operator string may also include a global phase factor from the set {±1,±i}. The Pauli operators obey the following multiplication rules:

    X_{j}^2 = Y_{j}^2 = Z_{j}^2 = I  
    X_{j}Y_{j} = iZ_{j}, Y_{j}Z_{j} = iX_{j}, Z_{j}X_{j} = iY_{j} (and cyclic permutations)  
    Operators on the same qubit anticommute.  
    Operators on different qubits commute.  

    Your task is to compute the product of the two operator strings by applying the Pauli algebra. The result should be a single simplified operator string, combining operators acting on the same qubit where applicable, and simplifying all coefficients to a single phase factor (±1, ±i). You must preserve the order of qubit indices in ascending order in your final expression.

    Format your answer as a single operator string, with phase factor in front (e.g., -i X_{1}Y_{2}Z_{3}), omitting identity operators. Do not include explanatory text

In [38]:
answers

[PauliTerm(coefficient=(-1+0j), pauli_string={0: 'Y', 1: 'X', 2: 'Z', 3: 'X', 4: 'Z', 5: 'Z', 6: 'Y'}, text='- Y_{0} X_{1} Z_{2} X_{3} Z_{4} Z_{5} Y_{6}'),
 PauliTerm(coefficient=1j, pauli_string={0: 'Y', 1: 'Y', 3: 'Z', 4: 'Y', 5: 'Y', 6: 'Z', 7: 'Y'}, text='+i Y_{0} Y_{1} Z_{3} Y_{4} Y_{5} Z_{6} Y_{7}'),
 PauliTerm(coefficient=1j, pauli_string={2: 'Y', 5: 'Y', 7: 'Z'}, text='+i Y_{2} Y_{5} Z_{7}'),
 PauliTerm(coefficient=(-0-1j), pauli_string={0: 'X', 2: 'Z', 3: 'X', 4: 'Z', 5: 'Z', 6: 'X', 7: 'Y'}, text='-i X_{0} Z_{2} X_{3} Z_{4} Z_{5} X_{6} Y_{7}'),
 PauliTerm(coefficient=1j, pauli_string={0: 'X', 2: 'Z', 3: 'X', 5: 'Z', 6: 'Y', 7: 'Z'}, text='+i X_{0} Z_{2} X_{3} Z_{5} Y_{6} Z_{7}'),
 PauliTerm(coefficient=(1+0j), pauli_string={1: 'Y', 2: 'Y', 3: 'Z', 4: 'Y', 6: 'Y'}, text='Y_{1} Y_{2} Z_{3} Y_{4} Y_{6}'),
 PauliTerm(coefficient=1j, pauli_string={0: 'Y', 1: 'X', 2: 'Y', 4: 'Y', 7: 'Y'}, text='+i Y_{0} X_{1} Y_{2} Y_{4} Y_{7}'),
 PauliTerm(coefficient=(-0-1j), pauli_string={0: 'X'

In [39]:
LLM_answers = [
    '- X_{0} X_{1} Y_{2} Z_{3} Z_{4} Z_{5} Y_{6}',
    '+i X_{0} X_{1} Z_{0} Z_{1} Z_{3} Y_{4} Y_{5} Z_{6} Y_{7}',
    '+ Z_{2} Y_{5} Z_{7}',
    '- Y_{0} X_{2} Y_{3} Z_{4} Z_{5} Z_{6} Y_{7}',
    '+ X_{0} Z_{2} X_{3} Z_{5} Z_{6} Z_{7}',
    '- Y_{1} X_{2} Z_{3} Y_{4} Y_{6}',
    '+ X_{0} X_{1} Y_{2} Y_{4} Z_{6} Y_{7} Z_{0}',
    '- Y_{0} Z_{0} Y_{2} X_{3} Y_{6}',
    '+i Z_{4} Z_{5} Z_{6}',
    '+ X_{0} X_{1} X_{2} Y_{3} Z_{4} Z_{5} Z_{6} Z_{7}'
]

In [40]:
LLM_answers_pc = [mcp.PauliTerm(text = LLM_answers[i]) for i in range(len(LLM_answers))]
accuracy = [LLM_answers_pc[i] == answers[i] for i in range(min(len(LLM_answers), len(answers)))]
print(f'N_max = {N_max}, question_volume  = {iterations}, accuracy = {sum(accuracy)/len(accuracy)}')

N_max = 8, question_volume  = 10, accuracy = 0.0


________________________________________

In [41]:
N_max = 9
iterations = 10

questions, answers = generate_questions_and_answers(N_max, iterations)
print(questions)

You are given two strings of Pauli operators acting on qubits indexed by subscripts. Each operator string may also include a global phase factor from the set {±1,±i}. The Pauli operators obey the following multiplication rules:

    X_{j}^2 = Y_{j}^2 = Z_{j}^2 = I  
    X_{j}Y_{j} = iZ_{j}, Y_{j}Z_{j} = iX_{j}, Z_{j}X_{j} = iY_{j} (and cyclic permutations)  
    Operators on the same qubit anticommute.  
    Operators on different qubits commute.  

    Your task is to compute the product of the two operator strings by applying the Pauli algebra. The result should be a single simplified operator string, combining operators acting on the same qubit where applicable, and simplifying all coefficients to a single phase factor (±1, ±i). You must preserve the order of qubit indices in ascending order in your final expression.

    Format your answer as a single operator string, with phase factor in front (e.g., -i X_{1}Y_{2}Z_{3}), omitting identity operators. Do not include explanatory text

In [42]:
answers

[PauliTerm(coefficient=1j, pauli_string={0: 'Z', 2: 'Z', 3: 'Y', 4: 'Z', 5: 'X', 7: 'Z'}, text='+i Z_{0} Z_{2} Y_{3} Z_{4} X_{5} Z_{7}'),
 PauliTerm(coefficient=(-0-1j), pauli_string={3: 'Z', 4: 'Y', 5: 'Z', 6: 'X', 7: 'Z', 8: 'X'}, text='-i Z_{3} Y_{4} Z_{5} X_{6} Z_{7} X_{8}'),
 PauliTerm(coefficient=1j, pauli_string={1: 'X', 3: 'X', 4: 'Z', 5: 'Z', 6: 'Y'}, text='+i X_{1} X_{3} Z_{4} Z_{5} Y_{6}'),
 PauliTerm(coefficient=1j, pauli_string={0: 'Z', 1: 'X', 2: 'Y', 3: 'Y', 4: 'Z', 6: 'Z', 7: 'Z', 8: 'X'}, text='+i Z_{0} X_{1} Y_{2} Y_{3} Z_{4} Z_{6} Z_{7} X_{8}'),
 PauliTerm(coefficient=(1+0j), pauli_string={1: 'Y', 2: 'X', 3: 'X', 4: 'Z', 5: 'X', 6: 'X', 8: 'Z'}, text='Y_{1} X_{2} X_{3} Z_{4} X_{5} X_{6} Z_{8}'),
 PauliTerm(coefficient=(1+0j), pauli_string={0: 'Y', 1: 'X', 3: 'X', 5: 'X', 7: 'Y'}, text='Y_{0} X_{1} X_{3} X_{5} Y_{7}'),
 PauliTerm(coefficient=(-0-1j), pauli_string={0: 'X', 1: 'Y', 2: 'Z', 3: 'Y', 4: 'Z', 5: 'Y', 7: 'Y', 8: 'X'}, text='-i X_{0} Y_{1} Z_{2} Y_{3} Z_{4} Y

In [43]:
LLM_answers = [
    '- Y_{0} X_{0} Z_{2} Y_{3} Z_{4} X_{5} Z_{7}',
    '+i Y_{4} Z_{5} X_{6} Z_{8}',
    '- Y_{1} Z_{1} Y_{3} Z_{3} Z_{4} X_{6}',
    '+ X_{0} X_{1} Y_{2} Y_{3} Z_{6} Y_{7} Y_{8}',
    '-i Z_{1} X_{1} X_{2} X_{3} X_{4} X_{5} Z_{6} X_{8}',
    '- Y_{0} X_{0} X_{1} Z_{1} Z_{3} Y_{5} X_{6} Y_{7}',
    '- Y_{1} Z_{2} Y_{3} Z_{4} Y_{5} Y_{6} Z_{7} X_{8}',
    '+ X_{0} Y_{0} Z_{1} X_{2} Z_{6} Z_{7} Y_{8}',
    '+ Z_{0} Y_{1} Y_{2} Z_{3} X_{4} Y_{6} Y_{7} X_{5} Y_{8}',
    '- X_{0} X_{3} Z_{4} Z_{5} Z_{6} Y_{8}'
]

In [44]:
LLM_answers_pc = [mcp.PauliTerm(text = LLM_answers[i]) for i in range(len(LLM_answers))]
accuracy = [LLM_answers_pc[i] == answers[i] for i in range(min(len(LLM_answers), len(answers)))]
print(f'N_max = {N_max}, question_volume  = {iterations}, accuracy = {sum(accuracy)/len(accuracy)}')

N_max = 9, question_volume  = 10, accuracy = 0.0


________________________________________

In [45]:
N_max = 10
iterations = 10

questions, answers = generate_questions_and_answers(N_max, iterations)
print(questions)

You are given two strings of Pauli operators acting on qubits indexed by subscripts. Each operator string may also include a global phase factor from the set {±1,±i}. The Pauli operators obey the following multiplication rules:

    X_{j}^2 = Y_{j}^2 = Z_{j}^2 = I  
    X_{j}Y_{j} = iZ_{j}, Y_{j}Z_{j} = iX_{j}, Z_{j}X_{j} = iY_{j} (and cyclic permutations)  
    Operators on the same qubit anticommute.  
    Operators on different qubits commute.  

    Your task is to compute the product of the two operator strings by applying the Pauli algebra. The result should be a single simplified operator string, combining operators acting on the same qubit where applicable, and simplifying all coefficients to a single phase factor (±1, ±i). You must preserve the order of qubit indices in ascending order in your final expression.

    Format your answer as a single operator string, with phase factor in front (e.g., -i X_{1}Y_{2}Z_{3}), omitting identity operators. Do not include explanatory text

In [46]:
answers

[PauliTerm(coefficient=1j, pauli_string={3: 'Z', 4: 'Y', 5: 'Y', 6: 'X', 8: 'Y'}, text='+i Z_{3} Y_{4} Y_{5} X_{6} Y_{8}'),
 PauliTerm(coefficient=(-1+0j), pauli_string={0: 'Z', 1: 'Z', 2: 'Z', 3: 'Z', 5: 'Y', 6: 'Y', 7: 'Y', 8: 'X'}, text='- Z_{0} Z_{1} Z_{2} Z_{3} Y_{5} Y_{6} Y_{7} X_{8}'),
 PauliTerm(coefficient=(-1+0j), pauli_string={0: 'Y', 1: 'Y', 3: 'Z', 4: 'X', 5: 'Z', 6: 'Z', 8: 'Y', 9: 'Y'}, text='- Y_{0} Y_{1} Z_{3} X_{4} Z_{5} Z_{6} Y_{8} Y_{9}'),
 PauliTerm(coefficient=(-1+0j), pauli_string={0: 'X', 1: 'Y', 2: 'Z', 3: 'X', 4: 'Y', 5: 'X', 7: 'Y'}, text='- X_{0} Y_{1} Z_{2} X_{3} Y_{4} X_{5} Y_{7}'),
 PauliTerm(coefficient=(-1+0j), pauli_string={0: 'Y', 1: 'X', 4: 'Y', 6: 'Z', 7: 'Z', 8: 'X', 9: 'Z'}, text='- Y_{0} X_{1} Y_{4} Z_{6} Z_{7} X_{8} Z_{9}'),
 PauliTerm(coefficient=(1+0j), pauli_string={0: 'Z', 1: 'Z', 2: 'Y', 3: 'Z', 4: 'Y', 5: 'X', 6: 'Y', 9: 'X'}, text='Z_{0} Z_{1} Y_{2} Z_{3} Y_{4} X_{5} Y_{6} X_{9}'),
 PauliTerm(coefficient=(1+0j), pauli_string={0: 'X', 1: '

In [47]:
LLM_answers = [
    '+i Z_{3} Y_{4} Y_{5} X_{6} Y_{8}',
    '- X_{0} Y_{0} Z_{1} X_{2} Z_{3} X_{3} Z_{4} Y_{5} Y_{6} Y_{7}',
    '+i Y_{0} X_{1} Z_{1} X_{4} Z_{5} Z_{6} Z_{8} Y_{9}',
    '+ X_{0} X_{1} Z_{2} Y_{4} X_{5} Z_{6} Y_{7}',
    '+ X_{0} Y_{0} Y_{1} X_{5} Z_{6} Z_{8} X_{9}',
    '+ Y_{0} Z_{1} Y_{2} Z_{3} Y_{4} X_{5} Y_{6} X_{9}',
    '- Y_{0} X_{2} Z_{5} X_{6} Y_{7} Y_{8}',
    '+i Z_{0} Y_{1} Y_{2} Y_{3} Y_{5} X_{6} X_{7} Z_{8} Y_{9}',
    '- X_{0} Y_{1} Z_{2} X_{3} Y_{4} Z_{5} X_{6} X_{9}',
    '+ X_{1} X_{2} Z_{5} Y_{6} Y_{7} Z_{8} Y_{9}'
]

In [48]:
LLM_answers_pc = [mcp.PauliTerm(text = LLM_answers[i]) for i in range(len(LLM_answers))]
accuracy = [LLM_answers_pc[i] == answers[i] for i in range(min(len(LLM_answers), len(answers)))]
print(f'N_max = {N_max}, question_volume  = {iterations}, accuracy = {sum(accuracy)/len(accuracy)}')

N_max = 10, question_volume  = 10, accuracy = 0.1
